# dmpbridge — Experiment Log

Records current model experiments, prompting strategies, and results.

**Dataset:** 10 manually labeled DMP documents, 741 blocks total  
**Evaluation:** Block-level label accuracy and per-label F1 (see `notebooks/eval/` for full breakdowns)  
**Labels:** `title`, `section.title`, `section.description`, `question.text`, `answer.text`  
**Output directory:** `data/llmlabeled/`

## Experiment Registry

| ID | Model | Strategy | Output tag | Samples | Accuracy | Status |
|---|---|---|---|---|---|---|
| M01 | Claude Opus 4.8 | batch | `claude-opus-4-8_batch` | 10/10 | 94.9% | Done |
| M02 | Claude Opus 4.8 | whole-doc | `claude-opus-4-8_whole_doc` | 10/10 | 96.9% | Done |
| M03 | Llama 3.3 70B | batch | `llama3.3-70b_batch` | 10/10 | 94.1% | Done |
| M04 | Llama 3.3 70B | whole-doc | `llama3.3-70b_whole_doc` | 10/10 | 91.8% | Done |
| M05 | Llama 3.1 8B | batch | `llama3.1-8b_batch` | 10/10 | 67.3% | Done |
| M06 | Llama 3.1 8B | whole-doc | `llama3.1-8b_whole_doc` | 10/10 | 84.2% | Done |
| F01 | Llama 3.1 8B fine-tune | — | — | — | — | Planned |

Output files follow the pattern: `data/llmlabeled/sampleN_{output-tag}.json`  
Detailed per-label analysis: `notebooks/eval/` and `notebooks/experiments/01_prompt_strategy_comparison.ipynb`

---
## Prompting Strategies

### Batch

The document is classified in overlapping windows of 10 blocks. Each window carries 3 blocks of context from the previous window to preserve label continuity across boundaries. The model makes one API call per window.

- Context window: 10 blocks, 3-block overlap
- API calls per document: approximately total blocks / 7
- Provider: Ollama (local) or Anthropic API
- Inference: `dmpbridge` CLI (`dmpbridge/pipeline.py`, `dmpbridge/classifier.py`)

### Whole-document

All extracted blocks from the document are passed to the model in a single API call. The model classifies the entire document at once, with full visibility into structure and context.

- Context window: full document (typically 60–100 blocks)
- API calls per document: 1
- Provider: Ollama (local) or Anthropic API
- Inference: `run_wholedoc_ollama.py` (Ollama) or `run_wholedoc.py` (Anthropic)

---
## Claude Opus 4.8

**Provider:** Anthropic API  
**Model ID:** `claude-opus-4-8`  
**Input:** `data/pdfsamples/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/eval/claude-opus-4-8.ipynb`

### M01 — Batch

**Output files:** `data/llmlabeled/sampleN_claude-opus-4-8_batch.json`  
**Inference:** `dmpbridge` CLI with `--provider anthropic --model claude-opus-4-8`

| Metric | Value |
|---|---|
| Overall accuracy | 94.9% (703/741) |
| Samples complete | 10/10 |

Strong baseline across all label types. Detailed per-label F1 in eval notebook.

### M02 — Whole-document

**Output files:** `data/llmlabeled/sampleN_claude-opus-4-8_whole_doc.json`  
**Inference:** `python run_wholedoc.py`  
**Note:** Uses `max_tokens=16384` to accommodate full structured output.

| Metric | Value |
|---|---|
| Overall accuracy | 96.9% (718/741) |
| Samples complete | 10/10 |
| Delta vs batch | +2.0pp |

Full-document context improves `question.text` F1 by +26pp and `section.description` F1 by +15pp. The model handles structural ambiguity better when it can see surrounding sections. Whole-document is the recommended strategy for Claude Opus 4.8.

---
## Llama 3.3 70B

**Provider:** Ollama (local)  
**Model ID:** `llama3.3:70b` (Q4_K_M, ~42 GB)  
**Input:** `data/pdfsamples/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/eval/llama3.3-70b.ipynb`

### M03 — Batch

**Output files:** `data/llmlabeled/sampleN_llama3.3-70b_batch.json`  
**Inference:** `dmpbridge` CLI with `--provider ollama --model llama3.3:70b`

| Metric | Value |
|---|---|
| Overall accuracy | 94.1% (697/741) |
| Samples complete | 10/10 |

Competitive accuracy with Claude Opus 4.8 batch (−0.8pp). Runs fully offline. Detailed per-label F1 in eval notebook.

### M04 — Whole-document

**Output files:** `data/llmlabeled/sampleN_llama3.3-70b_whole_doc.json`  
**Inference:** `python run_wholedoc_ollama.py --model llama3.3:70b`

| Metric | Value |
|---|---|
| Overall accuracy | 91.8% (680/741) |
| Samples complete | 10/10 |
| Delta vs batch | −2.3pp |

Whole-document prompting hurts for this model. With full document context, the model over-classifies `question.text` blocks as `section.description`, producing a large drop in `question.text` F1 (−36pp). The batch strategy is the recommended approach for Llama 3.3 70B.

---
## Llama 3.1 8B

**Provider:** Ollama (local)  
**Model ID:** `llama3.1:8b` (Q4_K_M, ~5 GB)  
**Input:** `data/pdfsamples/sampleN.pdf` (10 files)  
**Evaluation notebook:** `notebooks/eval/llama3.1-8b.ipynb`

### M05 — Batch

**Output files:** `data/llmlabeled/sampleN_llama3.1-8b_batch.json`  
**Inference:** `dmpbridge` CLI with `--provider ollama --model llama3.1:8b`

| Metric | Value |
|---|---|
| Overall accuracy | 67.3% (499/741) |
| Samples complete | 10/10 |

Substantially below the 70B model. The 8B model cannot reliably distinguish `section.description` (funder-written) from `question.text` (researcher-written) in batch mode — few-shot examples are not sufficient to bridge this gap at this model size.

### M06 — Whole-document

**Output files:** `data/llmlabeled/sampleN_llama3.1-8b_whole_doc.json`  
**Inference:** `python run_wholedoc_ollama.py --model llama3.1:8b`

| Metric | Value |
|---|---|
| Overall accuracy | 84.2% (624/741) |
| Samples complete | 10/10 |
| Delta vs batch | +16.9pp |

Whole-document context produces a large accuracy gain for the 8B model (+16.9pp overall, +22pp `question.text` F1). Full document visibility compensates for the model's weaker semantic discrimination in batch mode. Even so, 84.2% is still 10pp below Llama 3.3 70B batch and 13pp below Claude whole-doc, and is not suitable for production use without fine-tuning.

---
## Cross-model Summary

| Model | Batch | Whole-doc | Delta | Recommendation |
|---|---|---|---|---|
| Claude Opus 4.8 | 94.9% | 96.9% | +2.0pp | Use whole-doc |
| Llama 3.3 70B | 94.1% | 91.8% | −2.3pp | Use batch |
| Llama 3.1 8B | 67.3% | 84.2% | +16.9pp | Neither — see F01 |

**Key findings:**

- Whole-document context helps Claude and small models, but hurts Llama 3.3 70B. The 70B model becomes over-confident about document-level structure and misclassifies individual block roles.
- Llama 3.3 70B batch is the best offline option at 94.1%, close to Claude batch (94.9%) at no API cost.
- Llama 3.1 8B whole-doc at 84.2% may be acceptable for low-resource deployments where accuracy requirements are relaxed, but is not recommended for production use.
- The accuracy gap between the best Llama configuration (94.1%) and Claude whole-doc (96.9%) is 2.8pp across 741 blocks — approximately 21 additional correct labels per 741.

Detailed per-label F1 and per-sample breakdowns: `notebooks/experiments/01_prompt_strategy_comparison.ipynb`  
Six-way dashboard: `notebooks/eval/_dashboard.ipynb`

---
## Future Plans

### F01 — Fine-tune Llama 3.1 8B

**Status:** Planned  
**Depends on:** Expanding the labeled dataset to at least 20 samples (need held-out evaluation set)

**Hypothesis:** Fine-tuning on labeled DMP blocks will close most of the accuracy gap between Llama 3.1 8B and Llama 3.3 70B, making low-resource deployment viable.

**Planned approach:**
- Generate block-level training pairs from `data/manuallabeled/`
- Fine-tune with LoRA or QLoRA
- Evaluate on a held-out set not used in training
- Target: `question.text` F1 from 33% (batch) to 70%+

**Blocker:** The current 10-sample dataset is too small to split into train/eval. Dataset expansion is a prerequisite.

---

*Other possible future work: sequence correction post-processing (state machine over label sequence); dataset expansion to non-NIH/NSF funders.*